# Phase 1 — Notebook 2: TF-IDF, Category Analysis & NMF

Builds feature matrices and runs all Phase 1 analytical work on the clean
corpus from `01_preprocess.ipynb`. `make_vec`, `add_bin`, and `group_key` —
all imported from `utils.py` — are shared across Steps 1, 3, and 4, which is
the main reason those three live together.

| Step | What | Key output |
|------|------|------------|
| 1 | Build TF-IDF matrices | `features/X_*.npz`, `vocab/vocab_*.csv` |
| 2 | Quality CP2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `nmf_weights.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |


## Setup

In [1]:
import sys, json, os, httpx
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import NMF
from collections import defaultdict

from openai import OpenAI

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (load_cfg, tokens_to_str, flat_freq, make_vec,
                   add_bin, group_key, quality_report)

CFG = load_cfg()

def OUT(subdir, fname):
    p = ROOT / "OUTPUTS" / subdir / fname
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

df  = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
ct  = CFG["tfidf"]
print(f"Loaded {len(df):,} projects  |  {df.shape[1]} columns")

Loaded 896,277 projects  |  45 columns


---
## Parameters

All tunable values live here. Edit this cell; do not edit parameter references in downstream cells.


In [2]:
# ── ANALYSIS PARAMETERS — tune here, do not edit downstream cells ────
GROUPBY_FIELD     = CFG["analysis"]["group_by"][0]
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
                        GROUPBY_FIELD,
                        f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose project data")
MIN_SHARED        = 3      # minimum shared NMF terms for cross-group theme detection
MIN_COVERAGE      = CFG["analysis"]["min_coverage"]   # minimum groups a theme must span
N_COMPONENTS      = CFG["nmf"]["n_components"]         # NMF topics per group
TOP_N_VOCAB       = 20     # top-N vocab terms fed into NMF per group
N_REPRESENTATIVE  = CFG["llm"]["n_representative_snippets"]

MODEL_LABELING        = "gpt-5.4-mini"   # per-topic labeling loop
MODEL_CURRENT_EVENTS  = "gpt-5.4"        # current events fetch (support step, not full reasoning)
MODEL_SYNTHESIS       = "gpt-5.4"        # cross-topic synthesis call

REVIEW_GROUP      = None   # set to a specific group value to pin; None = auto-selects largest

print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"GROUP_DESCRIPTION = {GROUP_DESCRIPTION!r}")


GROUPBY_FIELD     = 'project_category'
GROUP_DESCRIPTION = 'product categories teachers use to request classroom supplies on DonorsChoose'


---
## Step 1 — Build TF-IDF Matrices

One vectorizer fit per n-gram range on the full corpus. Trigrams are persisted
even if unused in Phase 1 analysis — Phase 2+ inherits them without re-extraction.


In [3]:
docs  = df["tokens"].apply(tokens_to_str).tolist()
# N-grams are generated here by sklearn rather than in preprocessing.
# The vocab CSVs written below are the persisted n-gram vocabulary the
# spec calls for — just produced at feature time instead of as a
# separate preprocessing step, which avoids a redundant intermediate artifact.
specs = {"X_unigram": (1,1), "X_bigram": (2,2), "X_unigram_bigram": (1,2)}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec            = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name]     = vec
    sz, nnz        = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz/(sz[0]*sz[1]):.3f}")

# Vocab artifact: feature names + IDF weights per matrix
for name, vec in vecs.items():
    (pd.DataFrame({"feature": vec.get_feature_names_out(), "idf": vec.idf_})
       .sort_values("idf")
       .to_csv(OUT("vocab", f"vocab_{name}.csv"), index=False))

feat_dir = ROOT / "OUTPUTS/features"
feat_dir.mkdir(parents=True, exist_ok=True)
for name, X in matrices.items():
    sp.save_npz(feat_dir / f"{name}.npz", X.tocsr())

META_COLS = ["project_id", "project_category", "posted_date", "funded_date",
             "metro_type_at_time_of_posting", "grade_band", "theme_version"]
meta = df[META_COLS].reset_index(drop=True)
try:    meta.to_parquet(feat_dir / "meta.parquet", index=False)
except ImportError: meta.to_csv(feat_dir / "meta.csv", index=False)

assert matrices["X_unigram"].shape[0] == len(df)

  X_unigram           : shape=(896277, 7296)  sparsity=0.991
  X_bigram            : shape=(896277, 1002220)  sparsity=1.000
  X_unigram_bigram    : shape=(896277, 1009516)  sparsity=1.000
  X_trigram           : shape=(896277, 730147)  sparsity=1.000


---
## Step 2 — Quality Report: Checkpoint 2


In [4]:
qr2 = quality_report(df, label="cp2", matrices=matrices,
                     save_path=OUT("quality", "quality_cp2.json"))


=======================================================  [cp2]
  Projects : 896,277
  Tok/proj : min=5  p50=70  max=228
  Vocab    : 7,438 unique tokens
  X_unigram           : shape=[896277, 7296]  sparsity=0.991
  X_bigram            : shape=[896277, 1002220]  sparsity=1.000
  X_unigram_bigram    : shape=[896277, 1009516]  sparsity=1.000
  X_trigram           : shape=[896277, 730147]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
bins       = CFG["analysis"].get("bins", [])
min_proj   = CFG["analysis"]["min_category_projects"]
top_n      = TOP_N_VOCAB

df_work    = add_bin(df, bins) if bins else df.copy()
df_work    = df_work.reset_index(drop=True)
group_cols = [GROUPBY_FIELD, "bin"] if bins else [GROUPBY_FIELD]

# Fit once; reuse across all slices
all_docs  = df_work["tokens"].apply(tokens_to_str).tolist()
vec_cat   = make_vec(2, 0.95, (1, 2))
X_full    = vec_cat.fit_transform(all_docs)
feat      = vec_cat.get_feature_names_out()
idf_vals  = vec_cat.idf_

def cat_tfidf_slice(idx):
    """Score category slice by index against the rest of the corpus."""
    rest_idx  = df_work.index.difference(idx)
    X_cat     = X_full[idx.tolist()]
    X_rest    = X_full[rest_idx.tolist()]
    cat_prev  = (X_cat  > 0).mean(axis=0).A1
    rest_prev = (X_rest > 0).mean(axis=0).A1 if len(rest_idx) else np.zeros(len(feat))
    tf        = X_cat.mean(axis=0).A1
    return (pd.DataFrame({
                "token":         feat,
                "tf":            tf,
                "idf":           idf_vals,
                "tfidf":         tf * idf_vals,
                "prevalence":    cat_prev,
                "contrast":      cat_prev - rest_prev,
                "project_count": (X_cat > 0).sum(axis=0).A1.astype(int),
            }).nlargest(top_n, "tfidf"))

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj: continue
    kd  = group_key(keys, group_cols)
    top = cat_tfidf_slice(sub.index)
    for col, val in kd.items(): top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(f"No groups met min_proj={min_proj} threshold — lower min_category_projects in params.yaml")
cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)
print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

300 rows  |  15 groups


,project_category,token,tf,idf,tfidf,prevalence,contrast,project_count
0,Art Supplies,art,0.019440,3.213304,0.062466,0.473504,0.384579,22526
1,Art Supplies,paint,0.012872,4.703714,0.060545,0.217287,0.203455,10337
2,Art Supplies,supply,0.014939,2.739492,0.040924,0.418326,0.256322,19901
3,Art Supplies,color,0.010647,3.648676,0.038849,0.229794,0.167966,10932
4,Art Supplies,creativity,0.010098,3.618401,0.036538,0.226704,0.162406,10785
5,Art Supplies,marker,0.008938,4.013061,0.035868,0.170979,0.128669,8134
6,Art Supplies,express,0.008640,4.059463,0.035075,0.170286,0.130289,8101
7,Art Supplies,cricut,0.005411,6.309174,0.034138,0.070628,0.069365,3360
8,Art Supplies,crayon,0.006763,5.014537,0.033916,0.102117,0.088779,4858
9,Art Supplies,creative,0.010727,3.114602,0.033410,0.277952,0.166087,13223


In [6]:
# Top contrast tokens per group — quick cross-group comparison
(cat_tfidf_df.groupby(group_cols[0])
    .apply(lambda g: g.nlargest(5, "contrast")["token"].tolist(), include_groups=False)
    .rename("top_contrast_tokens").to_frame())

,top_contrast_tokens
project_category,
Art Supplies,"[art, supply, create, paint, project]"
Awaiting Classification,"[ron clark, ron, clark, academy, clark academy]"
Books,"[book, read, library, reader, interest]"
Classroom Basics,"[supply, pencil, paper, marker, organize]"
Computers & Tablets,"[technology, ipad, access, computer, laptop]"
Educational Kits & Games,"[skill, hand, play, fun, game]"
Flexible Seating,"[seat, chair, flexible, comfortable, flexible ..."
"Food, Clothing & Hygiene","[snack, healthy, food, hungry, clean]"
Instructional Technology,"[headphone, printer, print, resource, pay]"


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per category so the dominant axis in one category
does not suppress signal in others. Topics are candidates for human review —
not stable theme definitions (that is Phase 2).

The weights DataFrame records which projects load most strongly on each topic;
Step 5 uses it to select representative snippets by NMF weight rather than
randomly.


In [7]:
cn    = CFG["nmf"]
n_rep = N_REPRESENTATIVE

def nmf_one(docs):
    """Fit NMF on one slice. Returns (topics_df, W) or (None, None)."""
    vec = make_vec(ct["min_df"], ct["max_df"], tuple(ct.get("ngram_range", [1, 1])))
    X   = vec.fit_transform(docs)
    if X.shape[1] < N_COMPONENTS: return None, None
    model = NMF(n_components=N_COMPONENTS, random_state=cn["random_seed"],
                init="nndsvd", max_iter=cn["max_iter"])
    W     = model.fit_transform(X)
    vocab = vec.get_feature_names_out()
    rows  = []
    for i, comp in enumerate(model.components_):
        idx = comp.argsort()[::-1][:cn["top_words"]]
        rows.append({"topic_id": i, "top_terms": vocab[idx].tolist(),
                     "top_weights": comp[idx].tolist()})
    return pd.DataFrame(rows), W

all_topics, all_weights = [], []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj: continue
    kd     = group_key(keys, group_cols)
    docs   = sub["tokens"].apply(tokens_to_str).tolist()
    pids   = sub["project_id"].tolist()
    topics, W = nmf_one(docs)
    if topics is None: continue
    for col, val in kd.items(): topics[col] = val
    all_topics.append(topics)
    for tid in range(W.shape[1]):
        for rank, idx in enumerate(W[:, tid].argsort()[::-1][:n_rep]):
            all_weights.append({**kd, "topic_id": tid, "project_id": pids[idx],
                                "weight": float(W[idx, tid]), "rank": rank})

if not all_topics:
    raise RuntimeError(f"No groups produced NMF topics — check N_COMPONENTS={N_COMPONENTS} vs vocab size, or lower min_category_projects")
topics_df  = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)
topics_df.to_csv(OUT("analysis", "nmf_topics.csv"),   index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)
print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")
topics_df.head(6)

180 topics across 15 groups


,topic_id,top_terms,top_weights,project_category
0,0,"[community, culture, celebrate, other, world, ...","[1.171411601144529, 0.800192138761936, 0.63379...",Art Supplies
1,1,"[problem, solve, problem solve, science, think...","[0.8618271569018859, 0.8223546785812131, 0.690...",Art Supplies
2,2,"[pencil, marker, crayon, glue, color, supply, ...","[1.1094576934433082, 1.032664132601684, 0.8611...",Art Supplies
3,3,"[motor, fine, fine motor, motor skill, skill, ...","[1.3883624602030842, 1.3322030165116807, 1.304...",Art Supplies
4,4,"[low, income, low income, family, income famil...","[1.473301759330101, 1.4474962519000616, 1.4258...",Art Supplies
5,5,"[support, foster, essential, tool, environment...","[0.7365426943405854, 0.6967021621304971, 0.597...",Art Supplies


In [8]:
_eligible_groups = topics_df[group_cols[0]].unique().tolist()
if REVIEW_GROUP is not None and REVIEW_GROUP not in _eligible_groups:
    raise ValueError(f"REVIEW_GROUP={REVIEW_GROUP!r} not in topics — eligible: {_eligible_groups}")
REVIEW_GROUP = REVIEW_GROUP or topics_df[group_cols[0]].value_counts().index[0]
(topics_df[topics_df[group_cols[0]] == REVIEW_GROUP]
    [["topic_id","top_terms"]]
    .assign(top_terms=lambda x: x.top_terms.apply(lambda t: ", ".join(t[:8]))))

,topic_id,top_terms
0,0,"community, culture, celebrate, other, world, s..."
1,1,"problem, solve, problem solve, science, thinke..."
2,2,"pencil, marker, crayon, glue, color, supply, b..."
3,3,"motor, fine, fine motor, motor skill, skill, p..."
4,4,"low, income, low income, family, income family..."
5,5,"support, foster, essential, tool, environment,..."
6,6,"emotional, social, social emotional, calm, emo..."
7,7,"cricut, machine, shirt, design, bulletin, bull..."
8,8,"read, writing, math, book, word, write, learne..."
9,9,"some, hard, thing, love, kid, work, class, one"


In [9]:
# Cross-group universal themes — terms appearing in NMF topics across many groups

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i+1:]:
        if ci == cj: continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

# Sort by n_groups desc, then drop any theme whose terms are a subset of a prior theme
rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE: continue
    if not any(terms <= prior for prior in seen):
        deduped.append({"theme": ", ".join(sorted(terms)[:5]),
                        "n_groups": len(cats),
                        "categories": sorted(cats)})
        seen.append(terms)

pd.DataFrame(deduped).reset_index(drop=True)

,theme,n_groups,categories
0,"income, low, low income",12,"[Art Supplies, Books, Classroom Basics, Comput..."
1,"free, free reduce, lunch, price, receive",9,"[Art Supplies, Classroom Basics, Computers & T..."
2,"problem, solve, thinke",8,"[Art Supplies, Classroom Basics, Computers & T..."
3,"family, income, low, low income",8,"[Art Supplies, Books, Classroom Basics, Comput..."
4,"emotional, self, social",8,"[Art Supplies, Books, Computers & Tablets, Edu..."
5,"free, free reduce, lunch, price, price lunch",8,"[Art Supplies, Classroom Basics, Computers & T..."
6,"family, home, income, low, low income",7,"[Art Supplies, Books, Classroom Basics, Comput..."
7,"breakfast, free, free reduce, lunch, price",7,"[Art Supplies, Classroom Basics, Computers & T..."
8,"household, income household, low income",6,"[Art Supplies, Books, Classroom Basics, Musica..."
9,"emotional, self, social, social emotional",6,"[Art Supplies, Books, Educational Kits & Games..."


---
## Step 5 — LLM Topic Labeling

One API call per topic on compressed input only — never raw essay text at scale.
Representative snippets are chosen by NMF weight (not random).

Prompt field limits (`top_unigrams`, `top_bigrams`, `top_nmf_terms`) are applied
as item counts. Parse failures are stored with enough metadata to debug later.

**Requires:** `pip install anthropic` + `ANTHROPIC_API_KEY` env var.
Without the package, stubs are written so downstream steps stay testable.


In [10]:
# Initiate OpenAI credentials

import os
import httpx
from openai import OpenAI

if "client" not in globals():
    from openai import OpenAI
    import httpx
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Missing OPENAI_API_KEY environment variable.")
    http_client = httpx.Client(verify=False)
    client = OpenAI(api_key=api_key, http_client=http_client)
    print("✅ OpenAI client initialized.")
else:
    print("🔁 Client already loaded; skipping setup.")

✅ OpenAI client initialized.


In [12]:
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a JSON object, no preamble, no markdown fences.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = a rhetorical framing signal injected during preprocessing "
    "(e.g. __framing_barrier_removal__ means essays removing obstacles to participation)\n"
    "  __cat_[name]__ = a subject matter category token (e.g. __cat_genetics__)\n"
    "  __sub_[name]__ = a subcategory token\n"
    "These are meaningful signals — incorporate them into your label and description.\n\n"
    "Labeling rules:\n"
    "- Prefer the most specific defensible label.\n"
    "- Preserve concrete signals such as named programs, vendors, pedagogies, student "
    "populations, classroom use cases, or rhetorical framing when present.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- When framing tokens are present, treat them as first-class evidence about how "
    "teachers are positioning the request, not just background metadata.\n"
    "- If a topic is mixed, say what subthemes are colliding rather than forcing an "
    "overly neat label."
)
PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"
    "Instructions:\n"
    "- proposed_label should be concrete and specific, ideally naming the request or "
    "intervention, the population or context, and the framing when supported.\n"
    "- description should explain what makes this topic distinct from nearby generic "
    "topics.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or "
    "population if the evidence supports it.\n"
    "- If the evidence is genuinely mixed, preserve that ambiguity instead of inventing "
    "a falsely precise label.\n\n"
    f"Return a JSON object with exactly these keys:\n"
    f"  {GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes\n"
    'coherence_flag must be one of: "coherent", "mixed", "redundant"'
)

lc = CFG["llm"]

has_essay = "essay" in df.columns
pid_text  = (df.set_index("project_id")["essay"].str[:300] if has_essay
             else df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40])))

def build_input(t_row):
    terms    = t_row["top_terms"]
    key_cols = [GROUPBY_FIELD] + (["bin"] if "bin" in t_row.index else [])
    mask     = weights_df["topic_id"] == t_row["topic_id"]
    for col in key_cols: mask &= weights_df[col] == t_row[col]
    rep_pids = (weights_df[mask].sort_values("weight", ascending=False)
                    ["project_id"].tolist()[:N_REPRESENTATIVE])
    return {
        "group_value":       t_row[GROUPBY_FIELD],
        "topic_id":  int(t_row["topic_id"]),
        "bin_line":  f"\nBin: {t_row['bin']}"
                     if "bin" in t_row.index and pd.notna(t_row.get("bin")) else "",
        "unigrams":  ", ".join([x for x in terms if " " not in x][:6]),
        "bigrams":   ", ".join([x for x in terms if " " in  x][:4]),
        "nmf_terms": ", ".join(terms[:10]),
        "snippets":  "\n".join(f"- {pid_text.get(p, '')}" for p in rep_pids),
    }

results = []
for _, t in topics_df.iterrows():
    inp  = build_input(t)
    resp = client.chat.completions.create(
        model=MODEL_LABELING,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": PROMPT.format(**inp)},
        ],
    )
    text = resp.choices[0].message.content.strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = {"raw": text, "parse_error": True, "model": MODEL_LABELING,
               "model": MODEL_LABELING,
               "timestamp": datetime.now().isoformat(),
               GROUPBY_FIELD: inp["group_value"], "topic_id": inp["topic_id"]}
    results.append(obj)
    print(f"  {inp['group_value']} / topic {inp['topic_id']} "
          f"→ {obj.get('proposed_label', '?')}")

with open(OUT("analysis", "llm_topic_labels.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\n{len(results)} labels saved")

  Art Supplies / topic 0 → Cultural celebration and identity-building art supplies for community heritage projects
  Art Supplies / topic 1 → STEAM problem-solving and engineering design kits for hands-on prototyping
  Art Supplies / topic 2 → Basic art and classroom consumables restock for kindergarten–elementary creative work
  Art Supplies / topic 3 → Fine-motor sensory art materials for early childhood handwriting readiness
  Art Supplies / topic 4 → Low-income, free-lunch, and single-parent student context framing for creative-art projects
  Art Supplies / topic 5 → Art supply kits for creativity, classroom organization, and confident participation
  Art Supplies / topic 6 → Sensory self-regulation and social-emotional coping tools for stress, anxiety, and mindfulness
  Art Supplies / topic 7 → Cricut Maker vinyl/heat-press projects for custom labels, bulletin boards, and personalized shirts
  Art Supplies / topic 8 → Multisensory literacy centers for bilingual kindergarten and ea

In [13]:
labels_df = pd.DataFrame(results)
labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD,"topic_id","proposed_label","coherence_flag","description"]]

,project_category,topic_id,proposed_label,coherence_flag,description
0,Art Supplies,0,Cultural celebration and identity-building art...,coherent,This topic centers on art supplies used to cel...
1,Art Supplies,1,STEAM problem-solving and engineering design k...,coherent,This topic centers on art-supply-style materia...
2,Art Supplies,2,Basic art and classroom consumables restock fo...,coherent,This topic centers on requests for fundamental...
3,Art Supplies,3,Fine-motor sensory art materials for early chi...,coherent,This topic is distinct because the requested a...
4,Art Supplies,4,"Low-income, free-lunch, and single-parent stud...",coherent,This topic is distinct for its strong socioeco...
...,...,...,...,...,...
175,Virtual Visitors,7,Virtual animal visitors and Lion King–linked b...,mixed,This topic centers on teachers requesting virt...
176,Virtual Visitors,8,Sensory-motor art and music therapy supports f...,coherent,This topic centers on projects for special edu...
177,Virtual Visitors,9,"Deaf, hard-of-hearing, and English learner ear...",coherent,This topic is centered on preschool and early ...
178,Virtual Visitors,10,Virtual mindfulness and social-emotional skill...,coherent,This topic centers on live or virtual visitor ...


## Current events context for synthesis — fetches once, caches for 7 days

In [28]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
import re

def clean_label(text):
    """Remove injected __token__ signals and replace with readable paraphrase."""
    return re.sub(r'__[a-z_]+__\s*', '', str(text)).strip()

def build_topic_lines(df, groupby_field, group=None):
    if group is not None:
        df = df[df[groupby_field] == group]
    return "\n".join(
        f"  {row[groupby_field]} | topic {row.topic_id} | "
        f"label: {clean_label(row.proposed_label)} | "
        f"coherence: {row.coherence_flag} | "
        f"description: {clean_label(row.description)}"
        for _, row in df.iterrows()
    )

def run_synthesis(prompt):
    resp = client.chat.completions.create(
        model=MODEL_SYNTHESIS,
        messages=[
            {"role": "system", "content": SYNTHESIS_SYSTEM},
            {"role": "user",   "content": prompt},
        ],
    )
    return resp.choices[0].message.content.strip()

# ── Cross-group ────────────────────────────────────────────────────────────

topic_lines = build_topic_lines(labels_df, GROUPBY_FIELD)

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, insight-driven briefings for internal strategy audiences. "
    "You do not pad your answers or state the obvious.\n\n"
    "Some topic labels may reference rhetorical framing signals (e.g. 'barrier removal "
    "framing', 'chronic scarcity language') — these come from injected tokens that "
    "categorize how teachers write, not just what they request. Treat these as "
    "meaningful signals about teacher rhetorical strategy.\n\n"
    "Prioritize nuance over novelty. Prefer fine-grained distinctions, concrete "
    "mechanisms, specific populations, and framing differences over broad thematic "
    "summaries.\n\n"
    "Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS'). "
    "Do not use markdown headers (##). Do not use bullet points.\n\n"
    "Grounding: when writing Framing Differences findings, the contrast must be "
    "traceable to explicit language in the topic label or description — not inferred "
    "from the topic's general subject matter or implied by category context. If you "
    "cannot quote or closely paraphrase specific label language to establish both sides "
    "of a contrast, do not make the finding."
)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose,
grouped by {GROUPBY_FIELD} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines}
Your task: identify the most specific, decision-useful patterns in this topic landscape.
Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific group and topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. RECURRING INTERNAL SPLITS — cases where the same subtheme distinction recurs independently
   inside multiple groups: for example, a consumables-as-depletion vs. consumables-as-environment
   split that appears in both Art Supplies and Classroom Basics. Only flag a split if it appears
   in at least two groups with meaningfully similar internal logic. Name the groups and specific
   topics each time. Explain why the recurrence matters — what it reveals about a structural
   pattern in how teachers frame requests across categories.

2. CROSS-GROUP SIGNATURES — a specific program, vendor, pedagogy, student need, or rhetorical
   frame that shows up as a recognizable topic in multiple groups. Name every group it appears
   in and explain why the pattern is meaningful.

3. FRAMING DIFFERENCES — cases where similar underlying needs are presented through meaningfully
   different rhetorical frames, populations, or classroom contexts. Explain what changes across
   groups and why that difference matters.

4. NOTABLE ABSENCES — a topic you would expect given the internal logic of a group's other
   topics, but that is missing. To qualify, name at least two specific topics already present
   in the same group that make the gap conspicuous from within. Do not reason from other groups.
   Do not flag an absence simply because the theme exists elsewhere. The gap must be visible
   and surprising given what is already in that group.

5. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting
   (stated circumstance: equipment failure, new curriculum adoption, never having had access,
   disaster or event-driven need, population change in the classroom) and how they frame that
   reason for a donor audience (urgency construction, outcome promise, student identity appeal,
   teacher credibility claim). Only flag a trigger or strategy where the topic label or
   description makes the circumstance or rhetorical logic explicit. Do not infer either from
   content alone. Name the group and topic every time. Where the same trigger or strategy
   recurs across multiple groups, note it as a cross-group pattern.

Format as five labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the group and topic label every time. Do not use bullet points.
""".strip()

synthesis_cross = run_synthesis(SYNTHESIS_PROMPT_CROSS_GROUP)
with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w") as f:
    f.write(synthesis_cross)
print("── CROSS-GROUP ──────────────────────────────────────────")
print(synthesis_cross)

# ── Per-group loop ─────────────────────────────────────────────────────────

groups = [g for g in labels_df[GROUPBY_FIELD].unique() if g != "Awaiting Classification"]
per_group_results = {}

for group in groups:
    topic_lines_for_group = build_topic_lines(labels_df, GROUPBY_FIELD, group=group)

    SYNTHESIS_PROMPT_PER_GROUP = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose
for a single group: {group} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines_for_group}
Your task: identify the most specific, decision-useful patterns within this group.
Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. SUBTHEME DISTINCTIONS — distinct needs, populations, pedagogies, use cases, or framing
   modes within this group. Highlight contrasts that would be lost in a broad summary of the
   category. Only flag a distinction if a reasonable analyst working from the category name
   alone would likely miss it or collapse it into a single need.

2. INTERNAL FRAMING DIFFERENCES — cases within this group where similar underlying needs are
   presented through meaningfully different rhetorical frames, populations, or classroom
   contexts. Explain what changes across topics and why that difference matters for how the
   need should be interpreted or funded.

3. NOTABLE ABSENCES — a topic you would expect given the internal logic of this group's other
   topics, but that is missing. To qualify, name at least two specific topics already present
   in this group that make the gap conspicuous. Do not reason from other groups. The gap must
   be visible and surprising from within this group alone.

4. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting
   within this group (stated circumstance: equipment failure, new curriculum adoption, never
   having had access, disaster or event-driven need, population change in the classroom) and
   how they frame that reason for a donor audience (urgency construction, outcome promise,
   student identity appeal, teacher credibility claim). Only flag a trigger or strategy where
   the topic label or description makes the circumstance or rhetorical logic explicit. Do not
   infer either from content alone. Name the specific topic every time.

Format as four labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the topic label every time. Do not use bullet points.
""".strip()

    result = run_synthesis(SYNTHESIS_PROMPT_PER_GROUP)
    per_group_results[group] = result
    with open(OUT("analysis", f"llm_synthesis_{group.replace(' ', '_').replace('&','and')}.txt"), "w") as f:
        f.write(result)
    print(f"── {group} ──────────────────────────────────────────")
    print(result)
    print()

── CROSS-GROUP ──────────────────────────────────────────
1. WITHIN-GROUP SUBTHEMES

Art Supplies splits more sharply than a generic “art materials” view would suggest. Topic 1 (“STEAM problem-solving and engineering design kits for hands-on prototyping”) repurposes craft materials as engineering media, while topic 3 (“Fine-motor sensory art materials for early childhood handwriting readiness”) uses similar-looking materials for pre-writing muscle development, and topic 8 (“Multisensory literacy centers for bilingual kindergarten and early elementary learners”) uses art/manipulative materials for early literacy in multilingual settings. Category-level reporting would likely collapse these into general hands-on creativity, but the mechanisms are distinct: prototyping, motor remediation, and literacy instruction.

Art Supplies also contains a meaningful split between expressive production and classroom infrastructure. Topic 11 (“Expressive mixed-media art-making for student choice and se

In [ ]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────

# Combine all synthesis inputs
synthesis_input = "\n\n".join([
    f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}",
    *[f"=== {group} ===\n{text}" 
      for group, text in per_group_results.items()]
])

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, decision-facing briefings for external audiences: "
    "foundations, corporate partners, policymakers, and major donors. "
    "You will return ONLY valid JSON. No preamble, no explanation, no markdown fences."
)

INSIGHTS_PROMPT = f"""
You are given a comprehensive structured analysis of classroom project request patterns including:
within-group subthemes, cross-group signatures, framing differences, notable absences,
and request triggers and pitch strategies.

{synthesis_input}

Your task: produce a decision-facing insights document for external stakeholders —
executives, foundation leaders, state legislators, and major donors — who are
intelligent but not close to classroom realities. These readers should finish each
insight thinking: "I never would have known that, and the fact that this organization
can see it means I should take their priorities seriously."

SELECTION CRITERIA:
- ~8-15 cross-category insights (use judgment)
- 2-4 per-category insights for each major category
- Each insight must: surface something genuinely non-obvious about what classrooms need,
  reframe how a thoughtful outsider should interpret teacher demand, and have a clear
  implication for where attention or resources should go
- Prioritize insights that reveal hidden scale, misallocated attention, or a shift in
  how teachers are operating that external stakeholders would not expect
- Avoid: methodological commentary, insights that only matter to data practitioners,
  observations that confirm what the category label already suggests

INSIGHT STRUCTURE — for every insight use exactly these fields:
- title: a crisp, declarative headline that makes the finding feel like a reveal —
  not a description. Should read like a finding, not a topic label.
- what_seeing: 2-4 sentences describing the observed pattern in plain, concrete language.
  Write about what teachers are actually doing or asking for — not about the data.
  Make it vivid enough that a reader who has never seen a classroom request can picture it.
- why_easy_to_miss: 1-2 sentences explaining why this pattern is invisible to someone
  looking at surface-level categories or conventional reporting. Keep it tight —
  this is the "aha" setup, not the payoff.
- what_it_means: the payoff. 2-4 sentences that translate the pattern into consequence.
  This should land like a strategic implication, not an analytical note. Avoid
  abstract hedging. Write toward: what this means for where money goes, what gets
  underfunded by accident, or what this organization is uniquely positioned to address.
  The reader should feel informed and compelled, not instructed.
- source_topics: list of strings in format "group|topic_id" identifying the specific
  topics that ground this insight (e.g. ["Art Supplies|6", "Books|10", "Classroom Basics|5"])
- section: one of "key_insights", "appendix", or a category name matching the group exactly

STYLE:
- Direct, confident, and human — not academic, not corporate
- No mention of "topics", "model", "NMF", "pipeline", "cluster", "analysis", or any
  analytical machinery whatsoever
- No hedging phrases like "may suggest" or "could indicate" — write declaratively
- No filler. Every sentence should either reveal something or sharpen the implication.
- Do not repeat the same implication across insights

Return a JSON object with this structure:
{{
  "key_insights": [/* 8-15 insight objects */],
  "appendix": [/* secondary insight objects */],
  "by_category": {{
    "<category name>": [/* 2-4 insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user",   "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = resp.choices[0].message.content.strip()
insights_data = json.loads(raw_json)

with open(OUT("analysis", "insights_structured.json"), "w") as f:
    json.dump(insights_data, f, indent=2)

print(f"Key insights:  {len(insights_data.get('key_insights', []))}")
print(f"Appendix:      {len(insights_data.get('appendix', []))}")
print(f"Categories:    {list(insights_data.get('by_category', {}).keys())}")

In [ ]:
# ── BUILD DOCX + TRACEABILITY ──────────────────────────────────────────────
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd

# ── Load project-topic bridge ──────────────────────────────────────────────
# Reuse if already in memory, otherwise load from file
if "project_topic_bridge_df" not in dir():
    project_topic_bridge_df = pd.read_csv(OUT("analysis", "project_topic_bridge.csv"))

# ── Traceability: map source_topics → project_ids ─────────────────────────
def get_project_ids(source_topics, bridge_df, max_ids=50):
    """Return up to max_ids project_ids for a list of 'group|topic_id' strings."""
    if not source_topics:
        return []
    records = []
    for src in source_topics:
        try:
            group, tid = src.rsplit("|", 1)
            topic_key = f"{GROUPBY_FIELD}={group}|topic={tid}"
            matches = bridge_df[bridge_df["topic_key"] == topic_key]["project_id"].tolist()
            records.extend(matches)
        except ValueError:
            continue
    seen = list(dict.fromkeys(records))  # dedupe preserving order
    return seen[:max_ids]

# Build flat traceability table
traceability_rows = []
all_insights = (
    [("key_insights", i) for i in insights_data.get("key_insights", [])]
    + [("appendix",    i) for i in insights_data.get("appendix", [])]
    + [(cat, i)
       for cat, items in insights_data.get("by_category", {}).items()
       for i in items]
)

for section, insight in all_insights:
    pids = get_project_ids(insight.get("source_topics", []), project_topic_bridge_df)
    for pid in pids:
        traceability_rows.append({
            "section":       section,
            "insight_title": insight.get("title", ""),
            "source_topics": "; ".join(insight.get("source_topics", [])),
            "project_id":    pid,
        })

traceability_df = pd.DataFrame(traceability_rows)
traceability_path = OUT("analysis", "insights_traceability.csv")
traceability_df.to_csv(traceability_path, index=False)
print(f"Traceability rows: {len(traceability_df):,}  →  {traceability_path}")

# ── Helpers ────────────────────────────────────────────────────────────────
def add_heading(doc, text, level):
    p = doc.add_heading(text, level=level)
    p.runs[0].font.color.rgb = RGBColor(0, 0, 0)
    return p

def add_insight(doc, insight):
    """Write one insight block into the document."""
    # Title
    p = doc.add_paragraph()
    run = p.add_run(insight.get("title", ""))
    run.bold = True
    run.font.size = Pt(11)

    # Three labeled fields
    for label, key in [
        ("What we're seeing:",          "what_seeing"),
        ("Why this is easy to miss:",   "why_easy_to_miss"),
        ("What this means for funders:","what_it_means"),
    ]:
        p = doc.add_paragraph()
        label_run = p.add_run(label + "  ")
        label_run.bold = True
        label_run.font.size = Pt(10)
        body_run = p.add_run(insight.get(key, ""))
        body_run.font.size = Pt(10)
        p.paragraph_format.space_after = Pt(2)

    doc.add_paragraph()  # spacer

# ── Build document ─────────────────────────────────────────────────────────
doc = Document()

# Page margins
for section in doc.sections:
    section.top_margin    = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin   = Inches(1)
    section.right_margin  = Inches(1)

# Default font
style = doc.styles["Normal"]
style.font.name = "Arial"
style.font.size = Pt(10)

# ── Cross-category section ─────────────────────────────────────────────────
add_heading(doc, "Investigated Across Project Categories", level=1)
add_heading(doc, "Key Insights", level=2)

for insight in insights_data.get("key_insights", []):
    add_insight(doc, insight)

if insights_data.get("appendix"):
    add_heading(doc, "Appendix", level=2)
    for insight in insights_data["appendix"]:
        add_insight(doc, insight)

# ── By-category sections ───────────────────────────────────────────────────
add_heading(doc, "By Category", level=1)

for category, items in insights_data.get("by_category", {}).items():
    add_heading(doc, category, level=2)
    for insight in items:
        add_insight(doc, insight)

# ── Save ───────────────────────────────────────────────────────────────────
docx_path = OUT("analysis", "R_I_Trends_generated.docx")
doc.save(docx_path)
print(f"Docx saved → {docx_path}")
print(f"Open and compare against R_I_Trends__Prototype_Checkpoint__1_.docx")

In [27]:
from datetime import datetime, timedelta

ce_dir   = OUT("analysis", "placeholder").parent
ce_files = sorted(ce_dir.glob("current_events_*.txt"))
current_events = None

if ce_files:
    newest = ce_files[-1]
    ts_str = newest.stem.replace("current_events_", "")
    try:
        ts = datetime.strptime(ts_str, "%Y%m%d_%H%M%S")
        if datetime.now() - ts < timedelta(days=7):
            current_events = newest.read_text()
            print(f"Using cached current events from {ts.strftime('%Y-%m-%d %H:%M')} → {newest.name}")
    except ValueError:
        pass

if current_events is None:
    print("Fetching current events via web search...")
    ce_resp = client.chat.completions.create(
        model=MODEL_CURRENT_EVENTS,
        messages=[
            {"role": "system", "content": (
                "You are a policy researcher with knowledge of US K-12 education developments "
                "through early 2026. Be specific and factual. Cite named legislation, programs, "
                "or events where possible."
            )},
            {"role": "user", "content": (
                "Summarize recent developments (2025-2026) in these three areas as they relate "
                "to K-12 public schools in the United States: "
                "(1) Food insecurity and economic hardship affecting families with school-age children — "
                "including federal food assistance programs, SNAP, free and reduced lunch policy, "
                "and any broader economic pressures (inflation, unemployment, cost of living) "
                "that may affect whether students come to school fed and ready to learn. "
                "(2) AI tools, devices, and platforms in K-12 classrooms — new products, "
                "district policies, and debates about usage. "
                "(3) Student mental health funding and services — federal or state funding "
                "changes, school counselor ratios, and crisis intervention programs. "
                "Write 3-5 sentences per topic. Be specific, not general."
            )},
        ],
    )
    raw = ce_resp.choices[0].message.content
    current_events = (
        raw if isinstance(raw, str)
        else "\n\n".join(
            (b.get("text", "") if isinstance(b, dict) else getattr(b, "text", ""))
            for b in raw
            if (isinstance(b, dict) and b.get("type") == "text")
            or (hasattr(b, "type") and b.type == "text")
        )
    )
    ts_str  = datetime.now().strftime("%Y%m%d_%H%M%S")
    ce_path = ce_dir / f"current_events_{ts_str}.txt"
    ce_path.write_text(current_events)
    print(f"Saved → {ce_path.name}")

CONTEXT_SYSTEM = (
    "You are a senior education policy analyst. "
    "Your job is to add brief external context to an already completed synthesis "
    "without changing its findings.\n\n"
    "Do not revise, expand, or reinterpret the underlying findings. "
    "Only add concise contextual notes where a finding clearly intersects with "
    "recent external developments."
)

CONTEXT_PROMPT = f"""
Below is a completed synthesis derived from discovered topic clusters in DonorsChoose essays.

SYNTHESIS:
{synthesis}

Below is recent external context relevant to K-12 public schools.

CURRENT EVENTS CONTEXT:
{current_events}

Task:
Add a short contextual companion note that maps any clearly relevant findings to recent
external developments.

Rules:
- Do not change the original findings.
- Do not introduce new findings.
- Do not infer unmet need, causality, or prevalence from the news context.
- Only comment where there is a clear connection between the existing synthesis and the
  current-events context.
- If a finding has no clear external analogue, ignore it.
- Keep the tone analytical and restrained.

Format:
Write 3 short sections:
1. Food / household economic pressure context
2. AI / classroom technology context
3. Mental health / student regulation context

Under each section, write 0-3 short paragraphs tied to specific findings from the synthesis.
If nothing clearly applies, write: "No clear external context to add."
""".strip()

context_resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": CONTEXT_SYSTEM},
        {"role": "user",   "content": CONTEXT_PROMPT},
    ],
)
synthesis_context = context_resp.choices[0].message.content.strip()

with open(OUT("analysis", "llm_synthesis_context_overlay.txt"), "w") as f:
    f.write(synthesis_context)

print(synthesis_context)

Using cached current events from 2026-03-20 11:33 → current_events_20260320_113311.txt
## 1. Food / household economic pressure context

The synthesis’s distinction within **Food, Clothing & Hygiene** between direct hunger relief (topic 2), concentration-oriented snacks framed as “brain food” (topic 8), and hygiene kits tied to attendance and dignity (topic 6) aligns with the 2025 policy environment in which school-based student support remained closely connected to household economic strain. Recent debates around **SNAP**, **WIC**, **NSLP/SBP**, **Community Eligibility Provision (CEP)**, and state **Universal Free School Meals** expansions provide relevant external context for why classroom-level requests may sit alongside formal nutrition programs, without changing the synthesis’s finding that these are distinct intervention models.

The same context is relevant to the synthesis’s emphasis that hunger-related requests are not reducible to one narrative. In 2025, districts continued t

In [18]:
# ============================================================
# Deterministic bridges:
#   1) project_id -> topic_key
#   2) topic_key  -> summary_id
#   3) project_id -> summary_id
#
# This can live entirely in a final cell.
# It recomputes full project-topic loadings because the earlier
# weights_df only stores representative projects for labeling.
# ============================================================

import json
import pandas as pd
import numpy as np

ASSIGNMENT_THRESHOLD = float(CFG["analysis"]["topic_assignment_threshold"])
                
def make_topic_key(group_value, topic_id):
    return f"{GROUPBY_FIELD}={group_value}|topic={int(topic_id)}"

def extract_text_content(content):
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            parts.append(block.get("text", ""))
        elif getattr(block, "type", None) == "text":
            parts.append(getattr(block, "text", ""))
    return "".join(parts)

def strip_json_fences(text):
    text = text.strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()
    return text

# ------------------------------------------------------------
# A) Recompute FULL project-topic loadings deterministically
# ------------------------------------------------------------
loading_rows = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd   = group_key(keys, group_cols)
    docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    topics_tmp, W = nmf_one(docs)
    if topics_tmp is None:
        continue

    row_sums = W.sum(axis=1)
    primary_topics = W.argmax(axis=1)

    group_value = kd[GROUPBY_FIELD]

    for i, pid in enumerate(pids):
        row_sum = float(row_sums[i])
        if row_sum <= 0:
            continue

        for tid in range(W.shape[1]):
            weight = float(W[i, tid])
            share  = weight / row_sum

            loading_rows.append({
                "project_id": pid,
                GROUPBY_FIELD: group_value,
                "topic_id": int(tid),
                "topic_key": make_topic_key(group_value, tid),
                "topic_weight": weight,
                "topic_share": share,
                "is_primary_topic": int(tid == int(primary_topics[i])),
                "above_threshold": bool(share >= ASSIGNMENT_THRESHOLD),
            })

project_topic_loadings_df = pd.DataFrame(loading_rows)

# Add labels / descriptions onto the project-topic loadings
topic_meta_df = (
    topics_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .merge(
        labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "description", "coherence_flag"]],
        on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
)

topic_meta_df["topic_key"] = topic_meta_df.apply(
    lambda r: make_topic_key(r[GROUPBY_FIELD], r["topic_id"]),
    axis=1,
)

project_topic_loadings_df = project_topic_loadings_df.merge(
    topic_meta_df[
        [GROUPBY_FIELD, "topic_id", "topic_key", "proposed_label", "description", "coherence_flag"]
    ].drop_duplicates(),
    on=[GROUPBY_FIELD, "topic_id", "topic_key"],
    how="left",
)

# Thresholded bridge = the version you will usually use downstream
project_topic_bridge_df = (
    project_topic_loadings_df[project_topic_loadings_df["above_threshold"]]
    .copy()
    .reset_index(drop=True)
)

# Save both the full matrix-like long table and the thresholded bridge
project_topic_loadings_df.to_csv(OUT("analysis", "project_topic_loadings.csv"), index=False)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)

print(f"Saved full project-topic loadings  → {OUT('analysis', 'project_topic_loadings.csv')}")
print(f"Saved thresholded project-topic bridge ({ASSIGNMENT_THRESHOLD:.2f}) → {OUT('analysis', 'project_topic_bridge.csv')}")

# ------------------------------------------------------------
# B) Build deterministic topic_key -> summary_id bridge
#
# Important note:
# - The MECHANICAL join is deterministic.
# - The summary generation is still LLM-authored.
# - To keep runs auditable, save the emitted JSON artifacts.
# ------------------------------------------------------------

summary_topics_df = topic_meta_df.copy()

# Exclude placeholder group only; keep coherent/mixed/redundant visible to the model
summary_topics_df = summary_topics_df[
    summary_topics_df[GROUPBY_FIELD].astype(str) != "Awaiting Classification"
].copy()

summary_topics_df["topic_key"] = summary_topics_df.apply(
    lambda r: make_topic_key(r[GROUPBY_FIELD], r["topic_id"]),
    axis=1,
)

summary_topics_df["description"] = summary_topics_df["description"].fillna("")
summary_topics_df["coherence_flag"] = summary_topics_df["coherence_flag"].fillna("unknown")
summary_topics_df["proposed_label"] = summary_topics_df["proposed_label"].fillna("")

topic_lines = "\n".join(
    summary_topics_df.apply(
        lambda r: (
            f"- topic_key: {r['topic_key']} | "
            f"group_value: {r[GROUPBY_FIELD]} | "
            f"label: {r['proposed_label']} | "
            f"coherence: {r['coherence_flag']} | "
            f"description: {r['description']}"
        ),
        axis=1,
    ).tolist()
)

SUMMARY_BRIDGE_SYSTEM = (
    "You are a senior analyst creating a structured bridge between discovered topics and "
    "high-level synthesized insights. Respond ONLY with valid JSON.\n\n"
    "Your job is not to invent new evidence. Every synthesized insight must be grounded in "
    "one or more explicit topic_key values from the provided list."
)

SUMMARY_BRIDGE_PROMPT = f"""
Below is a list of discovered topics for DonorsChoose teacher essays, grouped by {GROUPBY_FIELD}.

{topic_lines}

Return ONLY a JSON array.
Each element in the array must be an object with exactly these keys:
- section
- summary_text
- source_topic_keys

Rules:
- source_topic_keys must be a JSON array of topic_key strings copied EXACTLY from the list above.
- Every summary must be grounded in 1 or more source_topic_keys.
- Do not invent topic keys.
- Prefer specific, decision-useful insights over broad summaries.
- Summaries should be concise but substantive.
- It is okay for one topic_key to support multiple summaries if truly warranted.
- Use these section names exactly:
  1. WITHIN_GROUP_SUBTHEMES
  2. CROSS_GROUP_SIGNATURES
  3. FRAMING_DIFFERENCES

Output between 6 and 12 total summaries across the three sections.
""".strip()

summary_resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    temperature=0,
    messages=[
        {"role": "system", "content": SUMMARY_BRIDGE_SYSTEM},
        {"role": "user",   "content": SUMMARY_BRIDGE_PROMPT},
    ],
)

summary_raw = extract_text_content(summary_resp.choices[0].message.content)
summary_raw = strip_json_fences(summary_raw)
summary_objs = json.loads(summary_raw)

if isinstance(summary_objs, dict):
    # tolerate {"summaries": [...]} wrapper if it ever appears
    if "summaries" in summary_objs:
        summary_objs = summary_objs["summaries"]
    else:
        raise ValueError("Expected a JSON array (or a {'summaries': [...]} wrapper).")

summary_records_df = pd.DataFrame(summary_objs).copy()

required_cols = {"section", "summary_text", "source_topic_keys"}
missing = required_cols - set(summary_records_df.columns)
if missing:
    raise ValueError(f"Missing required columns from summary JSON: {missing}")

def normalize_topic_keys(x):
    if isinstance(x, list):
        return [str(v) for v in x if pd.notna(v) and str(v).strip()]
    return []

summary_records_df["source_topic_keys"] = summary_records_df["source_topic_keys"].apply(normalize_topic_keys)
summary_records_df["section"] = summary_records_df["section"].astype(str).str.strip()
summary_records_df["summary_text"] = summary_records_df["summary_text"].astype(str).str.strip()

# Deterministic summary_id assignment after sorting
section_order = {
    "WITHIN_GROUP_SUBTHEMES": 1,
    "CROSS_GROUP_SIGNATURES": 2,
    "FRAMING_DIFFERENCES": 3,
}
summary_records_df["section_order"] = summary_records_df["section"].map(section_order).fillna(99)
summary_records_df["topic_key_sort"] = summary_records_df["source_topic_keys"].apply(
    lambda xs: "|".join(sorted(xs)) if isinstance(xs, list) else ""
)

summary_records_df = (
    summary_records_df
    .sort_values(["section_order", "topic_key_sort", "summary_text"])
    .reset_index(drop=True)
)

summary_records_df["summary_id"] = [
    f"S{i:03d}" for i in range(1, len(summary_records_df) + 1)
]

# Save the raw JSON too, so the run is fully auditable
with open(OUT("analysis", "summary_records_raw.json"), "w") as f:
    json.dump(summary_objs, f, indent=2)

summary_records_export_df = summary_records_df[
    ["summary_id", "section", "summary_text", "source_topic_keys"]
].copy()

summary_records_export_df["source_topic_keys"] = summary_records_export_df["source_topic_keys"].apply(
    lambda xs: " || ".join(xs)
)

summary_records_export_df.to_csv(OUT("analysis", "summary_records.csv"), index=False)

# One row per summary-topic pair
summary_topic_bridge_df = (
    summary_records_df[["summary_id", "section", "summary_text", "source_topic_keys"]]
    .explode("source_topic_keys")
    .rename(columns={"source_topic_keys": "topic_key"})
    .dropna(subset=["topic_key"])
    .reset_index(drop=True)
)

# Add topic metadata
summary_topic_bridge_df = summary_topic_bridge_df.merge(
    summary_topics_df[
        ["topic_key", GROUPBY_FIELD, "topic_id", "proposed_label", "description", "coherence_flag"]
    ].drop_duplicates(),
    on="topic_key",
    how="left",
)

summary_topic_bridge_df.to_csv(OUT("analysis", "summary_topic_bridge.csv"), index=False)

print(f"Saved summary records           → {OUT('analysis', 'summary_records.csv')}")
print(f"Saved summary-topic bridge      → {OUT('analysis', 'summary_topic_bridge.csv')}")

# ------------------------------------------------------------
# C) Join them: project_id -> summary_id
# ------------------------------------------------------------
project_summary_bridge_df = (
    project_topic_bridge_df.merge(
        summary_topic_bridge_df[
            ["summary_id", "section", "summary_text", "topic_key"]
        ].drop_duplicates(),
        on="topic_key",
        how="inner",
    )
    .sort_values([GROUPBY_FIELD, "summary_id", "project_id", "topic_share"], ascending=[True, True, True, False])
    .reset_index(drop=True)
)

project_summary_bridge_df.to_csv(OUT("analysis", "project_summary_bridge.csv"), index=False)

print(f"Saved project-summary bridge    → {OUT('analysis', 'project_summary_bridge.csv')}")

# ------------------------------------------------------------
# D) Quick QA prints
# ------------------------------------------------------------
print("\n--- QA ---")
print(f"Full project-topic rows:        {len(project_topic_loadings_df):,}")
print(f"Thresholded project-topic rows: {len(project_topic_bridge_df):,}")
print(f"Summary records:                {len(summary_records_df):,}")
print(f"Summary-topic rows:             {len(summary_topic_bridge_df):,}")
print(f"Project-summary rows:           {len(project_summary_bridge_df):,}")
print(f"Unique projects in bridge:      {project_summary_bridge_df['project_id'].nunique():,}")

display(
    summary_topic_bridge_df.head(10)
)

display(
    project_summary_bridge_df[
        [GROUPBY_FIELD, "summary_id", "topic_key", "project_id", "topic_share", "proposed_label"]
    ].head(20)
)

Saved full project-topic loadings  → OUTPUTS/analysis/project_topic_loadings.csv
Saved thresholded project-topic bridge (0.20) → OUTPUTS/analysis/project_topic_bridge.csv
Saved summary records           → OUTPUTS/analysis/summary_records.csv
Saved summary-topic bridge      → OUTPUTS/analysis/summary_topic_bridge.csv
Saved project-summary bridge    → OUTPUTS/analysis/project_summary_bridge.csv

--- QA ---
Full project-topic rows:        9,116,292
Thresholded project-topic rows: 1,323,334
Summary records:                10
Summary-topic rows:             64
Project-summary rows:           757,232
Unique projects in bridge:      423,587


,summary_id,section,summary_text,topic_key,project_category,topic_id,proposed_label,description,coherence_flag
0,S001,WITHIN_GROUP_SUBTHEMES,Art Supplies splits into at least three distin...,project_category=Art Supplies|topic=2,Art Supplies,2,Basic art and classroom consumables restock fo...,This topic centers on requests for fundamental...,coherent
1,S001,WITHIN_GROUP_SUBTHEMES,Art Supplies splits into at least three distin...,project_category=Art Supplies|topic=11,Art Supplies,11,Expressive mixed-media art-making for student ...,This topic is centered on classroom requests f...,coherent
2,S001,WITHIN_GROUP_SUBTHEMES,Art Supplies splits into at least three distin...,project_category=Art Supplies|topic=3,Art Supplies,3,Fine-motor sensory art materials for early chi...,This topic is distinct because the requested a...,coherent
3,S001,WITHIN_GROUP_SUBTHEMES,Art Supplies splits into at least three distin...,project_category=Art Supplies|topic=8,Art Supplies,8,Multisensory literacy centers for bilingual ki...,This topic is defined by requests for hands-on...,coherent
4,S001,WITHIN_GROUP_SUBTHEMES,Art Supplies splits into at least three distin...,project_category=Art Supplies|topic=6,Art Supplies,6,Sensory self-regulation and social-emotional c...,This topic centers on requesting sensory and c...,coherent
5,S002,WITHIN_GROUP_SUBTHEMES,Books requests are not just for more titles; t...,project_category=Books|topic=0,Books,0,Low-income take-home reading books for home–sc...,This topic centers on book requests framed as ...,coherent
6,S002,WITHIN_GROUP_SUBTHEMES,Books requests are not just for more titles; t...,project_category=Books|topic=1,Books,1,Print-rich classroom library and independent r...,This topic centers on teachers requesting book...,coherent
7,S002,WITHIN_GROUP_SUBTHEMES,Books requests are not just for more titles; t...,project_category=Books|topic=3,Books,3,Library expansion of graphic novels and high-i...,This topic is about expanding classroom or sch...,coherent
8,S002,WITHIN_GROUP_SUBTHEMES,Books requests are not just for more titles; t...,project_category=Books|topic=6,Books,6,Decodable books and explicit phonics practice ...,This topic centers on early literacy materials...,coherent
9,S002,WITHIN_GROUP_SUBTHEMES,Books requests are not just for more titles; t...,project_category=Books|topic=9,Books,9,Bilingual and English-Learner Books for Newcom...,"This topic centers on requests for bilingual, ...",coherent


,project_category,summary_id,topic_key,project_id,topic_share,proposed_label
0,Art Supplies,S001,project_category=Art Supplies|topic=2,2714908,0.217738,Basic art and classroom consumables restock fo...
1,Art Supplies,S001,project_category=Art Supplies|topic=8,2717605,0.251869,Multisensory literacy centers for bilingual ki...
2,Art Supplies,S001,project_category=Art Supplies|topic=11,2717605,0.208574,Expressive mixed-media art-making for student ...
3,Art Supplies,S001,project_category=Art Supplies|topic=11,2791354,0.248020,Expressive mixed-media art-making for student ...
4,Art Supplies,S001,project_category=Art Supplies|topic=11,2797407,0.273260,Expressive mixed-media art-making for student ...
5,Art Supplies,S001,project_category=Art Supplies|topic=8,2801108,0.293944,Multisensory literacy centers for bilingual ki...
6,Art Supplies,S001,project_category=Art Supplies|topic=3,2811905,0.347953,Fine-motor sensory art materials for early chi...
7,Art Supplies,S001,project_category=Art Supplies|topic=11,2821660,0.277884,Expressive mixed-media art-making for student ...
8,Art Supplies,S001,project_category=Art Supplies|topic=2,2824807,0.211241,Basic art and classroom consumables restock fo...
9,Art Supplies,S001,project_category=Art Supplies|topic=8,2826703,0.388852,Multisensory literacy centers for bilingual ki...


In [22]:
project_summary_bridge_df.query("summary_id == 'S002'")["project_id"].nunique()


88219

In [23]:
project_summary_bridge_df.query(
    "summary_id == 'S002' and project_category == 'Books'"
)["project_id"].nunique()

88219

In [25]:
project_summary_bridge_df.query(
    "summary_id == 'S002' and project_category == 'Books'"
)["project_id"].drop_duplicates().tolist()

AttributeError: 'list' object has no attribute 'head'